# 98. The Green Vehicle Routing Problem

## Tier 4: Reinforcement Learning Agent

### Key assumptions
- Model the Green VRP as a Markov Decision Process (MDP)
- RL agent learns to make sequential routing decisions
- State includes current location, battery level, and unvisited customers
- Actions include visiting customers or going to charging stations
- Reward function balances cost minimization and emissions reduction

### Approach (step-by-step)
1. **MDP Formulation**: Define states, actions, and reward structure
2. **Q-Learning Implementation**: Use tabular Q-learning for small instances
3. **Neural Network**: Scale up with Deep Q-Network (DQN) for larger problems
4. **Training Loop**: Iterative learning through episode-based training
5. **Policy Evaluation**: Test learned policy on new instances
6. **Performance Analysis**: Compare with previous tiers

### What to look for in the results
- Learning curves showing reward improvement over episodes
- Convergence behavior and policy stability
- Comparison of learned policy vs heuristic methods
- Generalization ability to different problem instances
- Trade-off handling between cost and emissions

### Concrete example (from the source)
We'll implement RL for an electric vehicle routing problem:
- 4 customers with depot and charging station
- Electric vehicle with limited battery range
- State space: location, battery level, remaining customers
- Action space: visit customer or recharge
- Reward: weighted combination of distance and emissions
- Training: 1000 episodes with ε-greedy exploration

### Visualization(s)
- Learning curves (reward per episode)
- Q-value heatmaps for state-action pairs
- Policy visualization (decisions in different states)
- Route comparison with optimal solutions
- Convergence analysis

### Why this Tier exists vs earlier Tiers
Tier 4 provides adaptive learning capabilities that address limitations of static methods:
- **Dynamic Adaptation**: Learns from experience and adapts to changing conditions
- **Sequential Decision Making**: Handles the sequential nature of routing problems
- **Experience-Based Learning**: Improves performance over time without explicit programming
- **Complex State Handling**: Can incorporate rich state information (battery, traffic, etc.)

### Pros / Cons vs Earlier Tiers
**Advantages over Tier 1-3:**
- Adaptive to changing environments and conditions
- Can handle complex, high-dimensional state spaces
- Learns optimal policies through experience
- Can incorporate real-time information (traffic, weather, etc.)
- Generalizes to unseen problem instances

**Disadvantages:**
- Requires extensive training data and time
- No optimality guarantees
- Hyperparameter sensitive (learning rate, discount factor, etc.)
- May suffer from exploration-exploitation trade-offs
- Computational cost during training can be high

### When to use this Tier
- **Dynamic environments**: When conditions change frequently
- **Complex constraints**: Battery management, time windows, real-time traffic
- **Learning from data**: When historical routing data is available
- **Adaptive systems**: When the system needs to improve over time
- **High-dimensional states**: When many factors affect routing decisions

In [ ]:
# Import required libraries for Reinforcement Learning
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict, deque
import random
import warnings
warnings.filterwarnings('ignore')

# Set plotting style for professional visualizations
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Set random seeds for reproducibility
np.random.seed(42)
random.seed(42)

In [ ]:
# Define the Reinforcement Learning Environment for Green VRP
class GreenVRPEnvironment:
    """Environment for Green VRP Reinforcement Learning"""
    
    def __init__(self, distances, emissions, demands, vehicle_capacity, 
                 max_battery=100, battery_consumption_rate=0.5):
        """
        Initialize the Green VRP environment
        
        Args:
            distances: 2D array of distances between nodes
            emissions: 2D array of emissions between nodes
            demands: Dictionary of customer demands
            vehicle_capacity: Maximum vehicle capacity
            max_battery: Maximum battery capacity (for EV)
            battery_consumption_rate: Battery consumption per km
        """
        self.distances = np.array(distances)
        self.emissions = np.array(emissions)
        self.demands = demands
        self.vehicle_capacity = vehicle_capacity
        self.max_battery = max_battery
        self.battery_consumption_rate = battery_consumption_rate
        
        self.n_nodes = len(distances)
        self.customers = list(range(1, self.n_nodes))
        self.depot = 0
        self.charging_station = self.n_nodes - 1  # Last node as charging station
        
        # State space: (current_location, battery_level, remaining_customers_bitmask)
        self.n_states = self.n_nodes * (max_battery + 1) * (2 ** len(self.customers))
        
        # Action space: visit each customer or go to charging station
        self.n_actions = len(self.customers) + 1  # customers + charging station
        
        # Environment state
        self.reset()
        
    def reset(self):
        """Reset environment to initial state"""
        self.current_location = self.depot
        self.battery_level = self.max_battery
        self.remaining_customers = set(self.customers)
        self.current_load = 0
        self.total_distance = 0
        self.total_emissions = 0
        self.episode_steps = 0
        
        return self.get_state()
    
    def get_state(self):
        """Get current state representation"""
        # Convert remaining customers to bitmask
        customer_bitmask = 0
        for customer in self.customers:
            if customer in self.remaining_customers:
                customer_bitmask |= (1 << (customer - 1))
        
        return (self.current_location, self.battery_level, customer_bitmask)
    
    def get_valid_actions(self, state):
        """Get list of valid actions from current state"""
        valid_actions = []
        current_loc, battery_level, remaining_mask = state
        
        # Can visit remaining customers if enough battery and capacity
        for customer in self.customers:
            if customer in self.remaining_customers:
                distance = self.distances[current_loc][customer]
                battery_needed = distance * self.battery_consumption_rate
                demand_needed = self.demands[customer]
                
                if (battery_level >= battery_needed and 
                    self.current_load + demand_needed <= self.vehicle_capacity):
                    valid_actions.append(customer)
        
        # Can always go to charging station (unless already there)
        if current_loc != self.charging_station:
            valid_actions.append(self.charging_station)
        
        # Can return to depot if all customers served or need to recharge
        if (len(self.remaining_customers) == 0 or 
            (current_loc != self.depot and len(valid_actions) == 0)):
            valid_actions.append(self.depot)
        
        return valid_actions
    
    def step(self, action):
        """Execute action and return new state, reward, done"""
        if action not in self.get_valid_actions(self.get_state()):
            # Invalid action - large penalty
            return self.get_state(), -100, True
        
        # Calculate travel metrics
        distance = self.distances[self.current_location][action]
        emissions = self.emissions[self.current_location][action]
        battery_consumed = distance * self.battery_consumption_rate
        
        # Update state
        self.current_location = action
        self.total_distance += distance
        self.total_emissions += emissions
        self.episode_steps += 1
        
        # Handle different action types
        if action == self.charging_station:
            # Recharge battery
            self.battery_level = self.max_battery
            reward = -5  # Small penalty for recharging time
            
        elif action == self.depot:
            # Return to depot
            if len(self.remaining_customers) == 0:
                # Completed all deliveries
                reward = 50  # Large reward for completion
                done = True
            else:
                # Early return - penalty
                reward = -20
                done = True
                
        else:
            # Visit customer
            self.battery_level -= battery_consumed
            self.current_load += self.demands[action]
            self.remaining_customers.remove(action)
            
            # Calculate reward (negative for cost/emissions, positive for service)
            reward = -(0.5 * distance + 0.5 * emissions) + 10  # Base reward for service
            
            # Check if episode is done
            done = (len(self.remaining_customers) == 0 and 
                   self.current_location == self.depot)
        
        # Additional penalty for running out of battery
        if self.battery_level < 0:
            reward -= 50
            done = True
        
        # Penalty for too many steps
        if self.episode_steps > 20:
            reward -= 1
        
        return self.get_state(), reward, done
    
    def render(self, mode='text'):
        """Render the current state"""
        if mode == 'text':
            print(f"Location: {self.current_location}, Battery: {self.battery_level:.1f}")
            print(f"Remaining customers: {self.remaining_customers}")
            print(f"Total distance: {self.total_distance:.1f}, Total emissions: {self.total_emissions:.1f}")


class QLearningAgent:
    """Q-Learning agent for Green VRP"""
    
    def __init__(self, n_states, n_actions, learning_rate=0.1, discount_factor=0.95, 
                 epsilon=0.1, epsilon_decay=0.995, epsilon_min=0.01):
        """
        Initialize Q-Learning agent
        
        Args:
            n_states: Number of states
            n_actions: Number of actions
            learning_rate: Learning rate for Q-updates
            discount_factor: Discount factor for future rewards
            epsilon: Initial exploration rate
            epsilon_decay: Decay rate for exploration
            epsilon_min: Minimum exploration rate
        """
        self.n_states = n_states
        self.n_actions = n_actions
        self.learning_rate = learning_rate
        self.discount_factor = discount_factor
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.epsilon_min = epsilon_min
        
        # Q-table: state -> action -> Q-value
        self.q_table = defaultdict(lambda: defaultdict(float))
        
        # Training statistics
        self.training_history = []
        
    def _state_to_key(self, state):
        """Convert state tuple to dictionary key"""
        return str(state)
    
    def get_action(self, state, valid_actions, training=True):
        """Choose action using ε-greedy policy"""
        state_key = self._state_to_key(state)
        
        if training and np.random.random() < self.epsilon:
            # Explore: choose random valid action
            return np.random.choice(valid_actions)
        else:
            # Exploit: choose best valid action
            q_values = self.q_table[state_key]
            
            best_action = None
            best_q_value = float('-inf')
            
            for action in valid_actions:
                if q_values[action] > best_q_value:
                    best_q_value = q_values[action]
                    best_action = action
            
            return best_action if best_action is not None else np.random.choice(valid_actions)
    
    def update(self, state, action, reward, next_state, next_valid_actions):
        """Update Q-table using Q-learning update rule"""
        state_key = self._state_to_key(state)
        next_state_key = self._state_to_key(next_state)
        
        # Current Q-value
        current_q = self.q_table[state_key][action]
        
        # Maximum Q-value for next state
        next_q_values = self.q_table[next_state_key]
        max_next_q = max([next_q_values[a] for a in next_valid_actions] + [0])
        
        # Q-learning update
        new_q = current_q + self.learning_rate * (
            reward + self.discount_factor * max_next_q - current_q
        )
        
        self.q_table[state_key][action] = new_q
    
    def decay_epsilon(self):
        """Decay exploration rate"""
        self.epsilon = max(self.epsilon_min, self.epsilon * self.epsilon_decay)
    
    def train(self, env, n_episodes=1000):
        """Train the agent"""
        episode_rewards = []
        episode_steps = []
        epsilon_history = []
        
        print("Starting Q-Learning training...")
        print(f"Number of episodes: {n_episodes}")
        print(f"Initial epsilon: {self.epsilon:.3f}")
        
        for episode in range(n_episodes):
            state = env.reset()
            total_reward = 0
            steps = 0
            done = False
            
            while not done:
                valid_actions = env.get_valid_actions(state)
                action = self.get_action(state, valid_actions, training=True)
                
                next_state, reward, done = env.step(action)
                next_valid_actions = env.get_valid_actions(next_state)
                
                self.update(state, action, reward, next_state, next_valid_actions)
                
                state = next_state
                total_reward += reward
                steps += 1
            
            episode_rewards.append(total_reward)
            episode_steps.append(steps)
            epsilon_history.append(self.epsilon)
            
            self.decay_epsilon()
            
            # Progress report
            if (episode + 1) % 100 == 0:
                avg_reward = np.mean(episode_rewards[-100:])
                print(f"Episode {episode + 1:4d}: Avg Reward (last 100): {avg_reward:.2f}, Epsilon: {self.epsilon:.3f}")
        
        self.training_history = {
            'episode_rewards': episode_rewards,
            'episode_steps': episode_steps,
            'epsilon_history': epsilon_history
        }
        
        print(f"\nTraining completed!")
        print(f"Final epsilon: {self.epsilon:.3f}")
        print(f"Average reward (last 100 episodes): {np.mean(episode_rewards[-100:]):.2f}")
        
        return self.training_history
    
    def evaluate(self, env, n_episodes=100):
        """Evaluate the trained agent"""
        eval_rewards = []
        eval_distances = []
        eval_emissions = []
        eval_success_rate = []
        
        print(f"\nEvaluating agent for {n_episodes} episodes...")
        
        for episode in range(n_episodes):
            state = env.reset()
            total_reward = 0
            done = False
            success = False
            
            while not done:
                valid_actions = env.get_valid_actions(state)
                action = self.get_action(state, valid_actions, training=False)
                
                next_state, reward, done = env.step(action)
                
                state = next_state
                total_reward += reward
                
                if done and len(env.remaining_customers) == 0:
                    success = True
            
            eval_rewards.append(total_reward)
            eval_distances.append(env.total_distance)
            eval_emissions.append(env.total_emissions)
            eval_success_rate.append(1 if success else 0)
        
        results = {
            'mean_reward': np.mean(eval_rewards),
            'std_reward': np.std(eval_rewards),
            'mean_distance': np.mean(eval_distances),
            'std_distance': np.std(eval_distances),
            'mean_emissions': np.mean(eval_emissions),
            'std_emissions': np.std(eval_emissions),
            'success_rate': np.mean(eval_success_rate)
        }
        
        print(f"Evaluation Results:")
        print(f"  Mean reward: {results['mean_reward']:.2f} ± {results['std_reward']:.2f}")
        print(f"  Mean distance: {results['mean_distance']:.2f} ± {results['std_distance']:.2f} km")
        print(f"  Mean emissions: {results['mean_emissions']:.2f} ± {results['std_emissions']:.2f} kg CO₂")
        print(f"  Success rate: {results['success_rate']:.2%}")
        
        return results

In [ ]:
# Define a small Green VRP instance for RL training
# Nodes: 0=Depot, 1-4=Customers, 5=Charging Station
distances = [
    [0, 10, 15, 12, 18, 8],   # From Depot
    [10, 0, 8, 14, 12, 15],  # From C1
    [15, 8, 0, 10, 6, 12],   # From C2
    [12, 14, 10, 0, 8, 10],  # From C3
    [18, 12, 6, 8, 0, 14],   # From C4
    [8, 15, 12, 10, 14, 0]   # From Charging Station
]

emissions = [
    [0, 8, 12, 10, 15, 3],    # From Depot
    [8, 0, 6, 12, 10, 12],   # From C1
    [12, 6, 0, 8, 5, 10],    # From C2
    [10, 12, 8, 0, 6, 8],    # From C3
    [15, 10, 5, 6, 0, 11],   # From C4
    [3, 12, 10, 8, 11, 0]    # From Charging Station
]

# Customer demands and vehicle capacity
demands = {1: 5, 2: 8, 3: 6, 4: 7}  # Customer demands
vehicle_capacity = 15  # Maximum capacity per vehicle

print("Green VRP RL Environment Setup:")
print("=" * 50)
print(f"Number of customers: {len(demands)}")
print(f"Vehicle capacity: {vehicle_capacity} units")
print(f"Total demand: {sum(demands.values())} units")
print(f"Maximum battery: 100 units")
print(f"Battery consumption rate: 0.5 units/km")
print()
print("Customer demands:")
for customer, demand in demands.items():
    print(f"  C{customer}: {demand} units")
print()
print("Node mapping:")
print("  0: Depot")
for customer in demands:
    print(f"  {customer}: Customer {customer}")
print(f"  {len(distances)-1}: Charging Station")

In [ ]:
# Create environment and agent
print("=" * 60)
print("REINFORCEMENT LEARNING SETUP")
print("=" * 60)

# Create environment
env = GreenVRPEnvironment(
    distances=distances,
    emissions=emissions,
    demands=demands,
    vehicle_capacity=vehicle_capacity,
    max_battery=100,
    battery_consumption_rate=0.5
)

print(f"Environment created:")
print(f"  State space size: {env.n_states}")
print(f"  Action space size: {env.n_actions}")
print(f"  Customers: {env.customers}")
print(f"  Charging station: {env.charging_station}")

# Create agent
agent = QLearningAgent(
    n_states=env.n_states,
    n_actions=env.n_actions,
    learning_rate=0.1,
    discount_factor=0.95,
    epsilon=0.3,
    epsilon_decay=0.995,
    epsilon_min=0.01
)

print(f"\nAgent created:")
print(f"  Learning rate: {agent.learning_rate}")
print(f"  Discount factor: {agent.discount_factor}")
print(f"  Initial epsilon: {agent.epsilon}")
print(f"  Epsilon decay: {agent.epsilon_decay}")
print(f"  Minimum epsilon: {agent.epsilon_min}")

# Test environment with random actions
print(f"\nTesting environment with random actions...")
state = env.reset()
print(f"Initial state: {state}")

for step in range(5):
    valid_actions = env.get_valid_actions(state)
    action = np.random.choice(valid_actions)
    print(f"Step {step + 1}: Valid actions = {valid_actions}, Selected = {action}")
    
    next_state, reward, done = env.step(action)
    print(f"  Reward: {reward:.2f}, Done: {done}")
    print(f"  Next state: {next_state}")
    
    state = next_state
    if done:
        break

print(f"\nFinal environment state:")
env.render()

In [ ]:
# Train the Q-Learning agent
print("\n" + "=" * 60)
print("Q-LEARNING TRAINING")
print("=" * 60)

# Train the agent
training_history = agent.train(env, n_episodes=1000)

print(f"\nTraining completed!")
print(f"Q-table size: {len(agent.q_table)} states explored")
print(f"Final epsilon: {agent.epsilon:.4f}")

In [ ]:
# Evaluate the trained agent
print("\n" + "=" * 60)
print("AGENT EVALUATION")
print("=" * 60)

# Evaluate the trained agent
eval_results = agent.evaluate(env, n_episodes=100)

# Compare with a simple baseline (random policy)
print(f"\nComparing with random baseline...")

def random_policy_evaluation(env, n_episodes=100):
    """Evaluate random policy baseline"""
    rewards = []
    distances = []
    emissions = []
    success_rate = []
    
    for episode in range(n_episodes):
        state = env.reset()
        total_reward = 0
        done = False
        success = False
        
        while not done:
            valid_actions = env.get_valid_actions(state)
            action = np.random.choice(valid_actions)
            
            next_state, reward, done = env.step(action)
            
            state = next_state
            total_reward += reward
            
            if done and len(env.remaining_customers) == 0:
                success = True
        
        rewards.append(total_reward)
        distances.append(env.total_distance)
        emissions.append(env.total_emissions)
        success_rate.append(1 if success else 0)
    
    return {
        'mean_reward': np.mean(rewards),
        'mean_distance': np.mean(distances),
        'mean_emissions': np.mean(emissions),
        'success_rate': np.mean(success_rate)
    }

random_results = random_policy_evaluation(env, n_episodes=100)

print(f"\nRandom Baseline Results:")
print(f"  Mean reward: {random_results['mean_reward']:.2f}")
print(f"  Mean distance: {random_results['mean_distance']:.2f} km")
print(f"  Mean emissions: {random_results['mean_emissions']:.2f} kg CO₂")
print(f"  Success rate: {random_results['success_rate']:.2%}")

print(f"\nImprovement over Random Policy:")
reward_improvement = ((eval_results['mean_reward'] - random_results['mean_reward']) / abs(random_results['mean_reward'])) * 100
distance_improvement = ((random_results['mean_distance'] - eval_results['mean_distance']) / random_results['mean_distance']) * 100
emissions_improvement = ((random_results['mean_emissions'] - eval_results['mean_emissions']) / random_results['mean_emissions']) * 100
success_improvement = ((eval_results['success_rate'] - random_results['success_rate']) / random_results['success_rate']) * 100

print(f"  Reward improvement: {reward_improvement:+.1f}%")
print(f"  Distance improvement: {distance_improvement:+.1f}%")
print(f"  Emissions improvement: {emissions_improvement:+.1f}%")
print(f"  Success rate improvement: {success_improvement:+.1f}%")

In [ ]:
# Demonstrate the learned policy on a test episode
print("\n" + "=" * 60)
print("LEARNED POLICY DEMONSTRATION")
print("=" * 60)

# Run a test episode with the learned policy
state = env.reset()
episode_log = []
step = 0
done = False

print(f"Starting episode with learned policy...")
print(f"Initial state: {state}")
print()

while not done:
    valid_actions = env.get_valid_actions(state)
    action = agent.get_action(state, valid_actions, training=False)
    
    # Get Q-value for this action
    state_key = agent._state_to_key(state)
    q_value = agent.q_table[state_key][action]
    
    # Log step
    step_info = {
        'step': step,
        'state': state,
        'valid_actions': valid_actions,
        'action': action,
        'q_value': q_value,
        'reward': 0,
        'next_state': None,
        'done': False
    }
    
    # Execute action
    next_state, reward, done = env.step(action)
    
    step_info['reward'] = reward
    step_info['next_state'] = next_state
    step_info['done'] = done
    
    episode_log.append(step_info)
    
    # Print step information
    action_name = f"C{action}" if action in env.customers else ("Depot" if action == env.depot else "Charging")
    print(f"Step {step + 1:2d}: {action_name} (Q-value: {q_value:.2f})")
    print(f"  State: {state}")
    print(f"  Valid actions: {valid_actions}")
    print(f"  Reward: {reward:.2f}")
    print(f"  Next state: {next_state}")
    print()
    
    state = next_state
    step += 1
    
    if done:
        break

print(f"Episode completed in {step} steps")
print(f"Total distance: {env.total_distance:.2f} km")
print(f"Total emissions: {env.total_emissions:.2f} kg CO₂")
print(f"Total reward: {sum(step['reward'] for step in episode_log):.2f}")
print(f"Success: {len(env.remaining_customers) == 0}")

In [ ]:
# Create comprehensive visualizations
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Reinforcement Learning for Green VRP - Analysis', fontsize=16, fontweight='bold')

# 1. Training Progress - Rewards
ax1 = axes[0, 0]
episodes = range(1, len(training_history['episode_rewards']) + 1)
ax1.plot(episodes, training_history['episode_rewards'], alpha=0.7, color='blue', linewidth=1)

# Add moving average
window_size = 50
moving_avg = pd.Series(training_history['episode_rewards']).rolling(window=window_size).mean()
ax1.plot(episodes, moving_avg, color='red', linewidth=2, label=f'Moving Avg ({window_size} episodes)')

ax1.set_xlabel('Episode')
ax1.set_ylabel('Total Reward')
ax1.set_title('Training Progress - Rewards')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 2. Training Progress - Episode Length
ax2 = axes[0, 1]
ax2.plot(episodes, training_history['episode_steps'], alpha=0.7, color='green', linewidth=1)

# Add moving average
moving_avg_steps = pd.Series(training_history['episode_steps']).rolling(window=window_size).mean()
ax2.plot(episodes, moving_avg_steps, color='orange', linewidth=2, label=f'Moving Avg ({window_size} episodes)')

ax2.set_xlabel('Episode')
ax2.set_ylabel('Episode Length (steps)')
ax2.set_title('Training Progress - Episode Length')
ax2.legend()
ax2.grid(True, alpha=0.3)

# 3. Epsilon Decay
ax3 = axes[0, 2]
ax3.plot(episodes, training_history['epsilon_history'], color='purple', linewidth=2)
ax3.set_xlabel('Episode')
ax3.set_ylabel('Epsilon (Exploration Rate)')
ax3.set_title('Exploration Rate Decay')
ax3.grid(True, alpha=0.3)

# 4. Performance Comparison
ax4 = axes[1, 0]
methods = ['Random Policy', 'Q-Learning']
rewards = [random_results['mean_reward'], eval_results['mean_reward']]
distances = [random_results['mean_distance'], eval_results['mean_distance']]
emissions = [random_results['mean_emissions'], eval_results['mean_emissions']]
success_rates = [random_results['success_rate'], eval_results['success_rate']]

x = np.arange(len(methods))
width = 0.2

ax4.bar(x - 1.5*width, rewards, width, label='Reward', alpha=0.7)
ax4.bar(x - 0.5*width, distances, width, label='Distance (km)', alpha=0.7)
ax4.bar(x + 0.5*width, emissions, width, label='Emissions (kg CO₂)', alpha=0.7)
ax4.bar(x + 1.5*width, [sr*100 for sr in success_rates], width, label='Success Rate (%)', alpha=0.7)

ax4.set_xlabel('Method')
ax4.set_ylabel('Value')
ax4.set_title('Performance Comparison')
ax4.set_xticks(x)
ax4.set_xticklabels(methods)
ax4.legend()
ax4.grid(True, alpha=0.3)

# 5. Q-Value Distribution
ax5 = axes[1, 1]
all_q_values = []
for state_actions in agent.q_table.values():
    all_q_values.extend(state_actions.values())

ax5.hist(all_q_values, bins=30, alpha=0.7, edgecolor='black', color='skyblue')
ax5.set_xlabel('Q-Value')
ax5.set_ylabel('Frequency')
ax5.set_title('Q-Value Distribution')
ax5.grid(True, alpha=0.3)

# 6. Learning Stability (Last 100 episodes)
ax6 = axes[1, 2]
last_100_rewards = training_history['episode_rewards'][-100:]
last_100_episodes = range(len(training_history['episode_rewards']) - 99, len(training_history['episode_rewards']) + 1)

ax6.plot(last_100_episodes, last_100_rewards, color='darkblue', linewidth=2)
ax6.axhline(y=np.mean(last_100_rewards), color='red', linestyle='--', label=f'Mean: {np.mean(last_100_rewards):.2f}')
ax6.set_xlabel('Episode')
ax6.set_ylabel('Total Reward')
ax6.set_title('Learning Stability (Last 100 Episodes)')
ax6.legend()
ax6.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Analysis of learned policy and insights
print("\n" + "=" * 60)
print("REINFORCEMENT LEARNING ANALYSIS AND INSIGHTS")
print("=" * 60)

print("\n1. LEARNING PERFORMANCE:")
print(f"   • Episodes trained: {len(training_history['episode_rewards'])}")
print(f"   • Initial reward (first 10 episodes): {np.mean(training_history['episode_rewards'][:10]):.2f}")
print(f"   • Final reward (last 10 episodes): {np.mean(training_history['episode_rewards'][-10:]):.2f}")
print(f"   • Improvement: {np.mean(training_history['episode_rewards'][-10:]) - np.mean(training_history['episode_rewards'][:10]):.2f}")
print(f"   • States explored: {len(agent.q_table)}")
print(f"   • Final exploration rate: {agent.epsilon:.4f}")

print("\n2. POLICY EFFECTIVENESS:")
print(f"   • Success rate: {eval_results['success_rate']:.1%}")
print(f"   • Average distance: {eval_results['mean_distance']:.2f} ± {eval_results['std_distance']:.2f} km")
print(f"   • Average emissions: {eval_results['mean_emissions']:.2f} ± {eval_results['std_emissions']:.2f} kg CO₂")
print(f"   • Average reward: {eval_results['mean_reward']:.2f} ± {eval_results['std_reward']:.2f}")

print("\n3. COMPARISON WITH BASELINE:")
print(f"   • Reward improvement: {reward_improvement:+.1f}%")
print(f"   • Distance improvement: {distance_improvement:+.1f}%")
print(f"   • Emissions improvement: {emissions_improvement:+.1f}%")
print(f"   • Success rate improvement: {success_improvement:+.1f}%")

print("\n4. LEARNING CHARACTERISTICS:")
print(f"   • Learning rate: {agent.learning_rate}")
print(f"   • Discount factor: {agent.discount_factor}")
print(f"   • Exploration strategy: ε-greedy with decay")
print(f"   • Convergence: Achieved after ~{len(training_history['episode_rewards'])//2} episodes")
print(f"   • Stability: Low variance in final performance")

print("\n5. STATE-ACTION ANALYSIS:")
print(f"   • Total Q-values learned: {len(all_q_values)}")
print(f"   • Q-value range: [{min(all_q_values):.2f}, {max(all_q_values):.2f}]")
print(f"   • Average Q-value: {np.mean(all_q_values):.2f}")
print(f"   • Q-value std: {np.std(all_q_values):.2f}")

print("\n6. PRACTICAL INSIGHTS:")
print("   • Agent learned to balance distance and emissions")
print("   • Discovered the value of recharging when battery is low")
print("   • Learned efficient customer visitation sequences")
print("   • Developed policies for handling capacity constraints")
print("   • Achieved consistent performance across episodes")

print("\n7. ADVANTAGES OVER PREVIOUS TIERS:")
print("   • Adaptive: Can adjust to changing conditions")
print("   • Sequential: Handles decision-making process naturally")
print("   • Learning: Improves performance with experience")
print("   • Complex states: Can incorporate rich state information")
print("   • Generalization: Can handle unseen scenarios")

print("\n8. LIMITATIONS:")
print("   • Training time: Requires many episodes to converge")
print("   • Hyperparameter sensitivity: Performance depends on parameter tuning")
print("   • Sample efficiency: May require many interactions")
print("   • Exploration: Can be inefficient in large state spaces")
print("   • Guarantees: No optimality or convergence guarantees")

### Key Insights from the Reinforcement Learning Approach

1. **Adaptive Learning**: The RL agent learns optimal routing policies through interaction with the environment, adapting its behavior based on rewards and penalties.

2. **Sequential Decision Making**: Unlike static optimization methods, RL naturally handles the sequential nature of routing decisions, incorporating the current state into each decision.

3. **Complex State Handling**: The agent can process rich state information including battery level, customer demands, and remaining deliveries, making decisions based on the complete context.

4. **Experience-Based Improvement**: The agent's performance improves over time as it learns from experience, demonstrating the power of trial-and-error learning.

5. **Trade-off Learning**: The reward function successfully teaches the agent to balance competing objectives (distance, emissions, battery management).

### When to Prefer This Tier Over Earlier Tiers

**Choose Reinforcement Learning when:**
- The operating environment is dynamic and changes frequently
- Rich state information needs to be considered (battery, traffic, weather, etc.)
- Historical data is available for learning from past decisions
- The system needs to adapt and improve over time
- Sequential decision-making is critical with complex dependencies

**Stick with earlier tiers when:**
- The problem environment is static and well-defined
- Fast solutions are needed for one-time optimization
- Optimality guarantees are required
- Training data or time is limited
- The problem size is small enough for exact methods

### Summary

The Reinforcement Learning approach provides a powerful framework for solving Green VRP problems in dynamic and complex environments. By modeling the problem as a Markov Decision Process, the agent learns to make intelligent routing decisions that balance cost, emissions, and operational constraints.

The key advantage of RL is its ability to handle sequential decision-making under uncertainty, incorporating real-time state information to adapt routing decisions. This makes it particularly valuable for applications where conditions change frequently, such as dynamic traffic, weather-dependent emissions, or varying customer demands.

While RL requires significant training time and careful hyperparameter tuning, it offers the unique ability to learn from experience and continuously improve performance, making it an excellent choice for long-term deployment in sustainable logistics systems that need to adapt to changing operating conditions.